In [ ]:
import os

DB_PATH = os.path.join(PROJECT_DIR, "news_articles.db")

if os.path.exists(DB_PATH):
    os.remove(DB_PATH)
    print("🗑️ Old DB deleted:", DB_PATH)
else:
    print("ℹ️ No existing DB found")

print("✅ Ready for fresh scrape")


NameError: name 'PROJECT_DIR' is not defined

In [ ]:
import os, sys, subprocess

REPO_URL = "https://github.com/EhsanulHaqueSiam/BDNewsPaperScraper.git"
PROJECT_DIR = os.path.abspath("BDNewsPaperScraper")

if not os.path.exists(PROJECT_DIR):
    subprocess.run(["git", "clone", REPO_URL, PROJECT_DIR], check=True)
else:
    subprocess.run(["git", "-C", PROJECT_DIR, "pull"], check=True)

pkgs = [
    "scrapy>=2.12.0",
    "scrapy-playwright>=0.0.42",
    "playwright>=1.49.0",
    "pandas>=2.2.0",
    "openpyxl>=3.1.0",
    "pytz",
    "python-dateutil",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + pkgs, check=True)
subprocess.run([sys.executable, "-m", "playwright", "install", "--with-deps", "chromium"], check=True)

print("✅ Setup done:", PROJECT_DIR)


✅ Setup done: /content/BDNewsPaperScraper


In [ ]:
import subprocess, shlex

cmd = f"cd {shlex.quote(PROJECT_DIR)} && scrapy list"
print("Running:", cmd)
subprocess.run(["bash", "-lc", cmd], check=True)


Running: cd /content/BDNewsPaperScraper && scrapy list


CompletedProcess(args=['bash', '-lc', 'cd /content/BDNewsPaperScraper && scrapy list'], returncode=0)

In [ ]:
from datetime import date

# 🔥 Last 6 months of 2025 (July → Dec 14)
start_date = date(2020, 1, 1)
end_date   = date(2020, 12, 30)

print("Scraping range:", start_date, "→", end_date)


Scraping range: 2020-01-01 → 2020-12-30


In [ ]:
import time, subprocess, shlex

print("\n" + "="*80)
print(f"🚀 Running spider: thedailystar ({start_date} → {end_date})")

try:
    cmd = (
        f"cd {shlex.quote(PROJECT_DIR)} && "
        f"python -m scrapy crawl thedailystar -L INFO "
        f"-a start_date={start_date} -a end_date={end_date}"
    )

    code = subprocess.call(["bash", "-lc", cmd])

    if code != 0:
        print(f"❌ thedailystar FAILED (exit code {code})")
    else:
        print(f"✅ thedailystar DONE")

except Exception as e:
    print(f"❌ thedailystar CRASHED:", e)

time.sleep(3)

print("\n🎉 DONE — Daily Star scraping finished.")



🚀 Running spider: thedailystar (2020-01-01 → 2020-12-30)
✅ thedailystar DONE

🎉 DONE — Daily Star scraping finished.


In [ ]:
import sqlite3, pandas as pd, os

DB_PATH = os.path.join(PROJECT_DIR, "news_articles.db")

with sqlite3.connect(DB_PATH) as conn:
    by_paper = pd.read_sql_query(
        """
        SELECT paper_name, COUNT(*) AS cnt
        FROM articles
        GROUP BY paper_name
        ORDER BY cnt DESC
        """,
        conn
    )

by_paper



,paper_name,cnt
0,thedailystar,2


In [ ]:
import sqlite3, pandas as pd, os

DB_PATH = os.path.join(PROJECT_DIR, "news_articles.db")

with sqlite3.connect(DB_PATH) as conn:
    df = pd.read_sql_query(
        "SELECT * FROM articles WHERE paper_name = 'thedailystar'",
        conn
    )

print("✅ Total Daily Star articles:", len(df))
df.head(5)




✅ Total Daily Star articles: 2


,id,url,paper_name,headline,article,publication_date,scraped_at
0,1,https://www.thedailystar.net/news/investigativ...,thedailystar,Toxic ships sail in on false papers,"""I've lost hope for life,"" he said. The diseas...","Sat Dec 19, 2020 12:00 AM",2025-12-23 06:19:57
1,2,https://www.thedailystar.net/news/investigativ...,thedailystar,Target forex reserve,Orion Group is the first business conglomerate...,"Wed Oct 28, 2020 12:00 AM",2025-12-23 06:20:04


In [ ]:
# 🔎 URL column auto-detect
url_col = next(c for c in df.columns if "url" in c.lower() or "link" in c.lower())
print("Using URL column:", url_col)

# 🔥 Daily Star business URL keyword
BUSINESS_KEYWORDS = [
    "thedailystar.net/business"
]

urls = df[url_col].astype(str).str.lower()

finance_df = df[urls.str.contains("thedailystar.net/business", na=False)].copy()

print("✅ Daily Star Business articles:", len(finance_df))

finance_df[[url_col]].head(10)


Using URL column: url
✅ Daily Star Business articles: 0


,url


In [ ]:
from google.colab import files
import os

# 🔥 Final output filename (clear & fixed)
OUT = "/content/dailystar_business_2025_July_Dec.csv"

# 🔐 Safety check
if finance_df is None or finance_df.empty:
    raise ValueError("❌ finance_df empty — nothing to export")

# Save CSV
finance_df.to_csv(OUT, index=False, encoding="utf-8-sig")

print("✅ Daily Star Business CSV saved at:", OUT)
print("📄 Total rows:", len(finance_df))
print("📄 Columns:", list(finance_df.columns))

# Download
files.download(OUT)


In [ ]:
## the business standard

In [ ]:
!pip install -q requests beautifulsoup4 pandas


In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from datetime import datetime
import time


In [ ]:

BASE = "https://www.tbsnews.net"

SECTIONS = {
    "economy": "/economy",
    "trade": "/trade",
    "banking": "/banking",
    "stock": "/stock-market",
    "corporate": "/corporate",
    "industry" : "/industry",
    "analysis": "/analysis",
    "bazaar": "/bazaar",
    "rmg": "/rmg",
    "aviation": "/aviation"
}

START_DATE = datetime(2021, 1, 1)
END_DATE   = datetime(2025, 12, 23)

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
}

In [ ]:
def extract_title(soup):
    h1 = soup.find("h1")
    if h1 and h1.get_text(strip=True):
        return h1.get_text(strip=True)

    meta = soup.find("meta", property="og:title")
    if meta and meta.get("content"):
        return meta["content"].strip()

    if soup.title:
        return soup.title.get_text(strip=True)

    return ""

def extract_pub_date_soft(soup):
    meta = soup.find("meta", property="article:published_time")
    if meta and meta.get("content"):
        d = pd.to_datetime(meta["content"], errors="coerce")
        if pd.notna(d):
            return d.date()
    return None   # fallback later

def extract_content(soup):
    content = " ".join(p.get_text(strip=True) for p in soup.select("article p"))
    if not content:
        content = " ".join(p.get_text(strip=True) for p in soup.select("p"))
    return content.strip()


In [ ]:
records = []
visited = set()

for section, path in SECTIONS.items():
    print(f"\n📂 SECTION: {section.upper()}")

    for page in range(0, 50):  # safe pagination
        url = f"{BASE}{path}?page={page}"
        print("Listing:", url)

        r = requests.get(url, headers=HEADERS, timeout=30)
        if r.status_code != 200 or len(r.text) < 1500:
            break

        soup = BeautifulSoup(r.text, "html.parser")

        links = set()
        for a in soup.select("a[href]"):
            href = a.get("href", "").strip()

            if href.startswith("/"):
                full = BASE + href
            elif href.startswith("http"):
                full = href
            else:
                continue

            if "tbsnews.net" not in full:
                continue

            # 🔥 relaxed section check
            if section not in full:
                continue

            links.add(full)

        if not links:
            break

        for link in links:
            if link in visited:
                continue
            visited.add(link)

            try:
                art = requests.get(link, headers=HEADERS, timeout=30)
                asoup = BeautifulSoup(art.text, "html.parser")

                title = extract_title(asoup)
                if not title:
                    continue

                pub_date = extract_pub_date_soft(asoup)
                if pub_date:
                    if pub_date < START_DATE.date() or pub_date > END_DATE.date():
                        continue
                else:
                    pub_date = None  # keep if date missing

                content = extract_content(asoup)
                if not content:
                    continue

                records.append({
                    "paper": "TheBusinessStandard",
                    "section": section,
                    "published_date": pub_date,
                    "title": title,
                    "url": link,
                    "content": content
                })

            except Exception:
                pass

            time.sleep(0.25)




📂 SECTION: ECONOMY
Listing: https://www.tbsnews.net/economy?page=0
Listing: https://www.tbsnews.net/economy?page=1
Listing: https://www.tbsnews.net/economy?page=2
Listing: https://www.tbsnews.net/economy?page=3
Listing: https://www.tbsnews.net/economy?page=4
Listing: https://www.tbsnews.net/economy?page=5
Listing: https://www.tbsnews.net/economy?page=6
Listing: https://www.tbsnews.net/economy?page=7
Listing: https://www.tbsnews.net/economy?page=8
Listing: https://www.tbsnews.net/economy?page=9
Listing: https://www.tbsnews.net/economy?page=10
Listing: https://www.tbsnews.net/economy?page=11
Listing: https://www.tbsnews.net/economy?page=12
Listing: https://www.tbsnews.net/economy?page=13
Listing: https://www.tbsnews.net/economy?page=14
Listing: https://www.tbsnews.net/economy?page=15
Listing: https://www.tbsnews.net/economy?page=16
Listing: https://www.tbsnews.net/economy?page=17
Listing: https://www.tbsnews.net/economy?page=18
Listing: https://www.tbsnews.net/economy?page=19
Listing: h

In [ ]:
df = pd.DataFrame(records)

print("\n✅ TOTAL TBS articles:", len(df))

if not df.empty and "section" in df.columns:
    print(df["section"].value_counts())
else:
    print("⚠️ No data scraped")

df.head()




✅ TOTAL TBS articles: 1846
section
aviation    608
industry    553
trade       395
economy     252
analysis     38
Name: count, dtype: int64


,paper,section,published_date,title,url,content
0,TheBusinessStandard,economy,2025-12-23,External debt falls $1.45b in three months,https://www.tbsnews.net/economy/external-debt-...,Bangladesh's external debt declined by $1.45 b...
1,TheBusinessStandard,economy,2025-12-23,Govt to procure 1.30 lakh tonnes of fertiliser,https://www.tbsnews.net/economy/govt-procure-1...,The interim government today (23 December) app...
2,TheBusinessStandard,economy,None,Industry,https://www.tbsnews.net/economy/industry,"TuesdayDecember 23, 2025 Experts have long bee..."
3,TheBusinessStandard,economy,None,Aviation,https://www.tbsnews.net/economy/aviation,"TuesdayDecember 23, 2025 According to a press ..."
4,TheBusinessStandard,economy,2025-12-23,Govt approves import of 1 lakh tonne rice from...,https://www.tbsnews.net/economy/govt-approves-...,"The government has approved several proposals,..."


In [ ]:
from google.colab import files

OUT = "/content/tbs_2021-2025_new_finance_articles.csv"

df.to_csv(OUT, index=False, encoding="utf-8-sig")

print("✅ CSV READY")
print("📄 File:", OUT)
print("📊 Rows:", len(df))

files.download(OUT)


✅ CSV READY
📄 File: /content/tbs_2021-2025_new_finance_articles.csv
📊 Rows: 1846


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>